In [ ]:
import argparse
import json
import logging
from pathlib import Path
from typing import Optional

from gliner2 import GLiNER2
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

EXTRACTION_CHARS = 6_000

SCHEMA = {
    "company": [
        
            "description::str::what the company does, the products or services it provides" ,
            "customers::str::who the company's customers are, if mentioned" ,
            "headquarters::str::where the company is headquartered, if mentioned" ,
            "founded::str::when the company was founded, if mentioned" ,
            "size::str::the size of the company (e.g. number of employees, revenue), if mentioned" ,
            "industry::str::the industry the company operates in, if mentioned"
        
    ]
}

# ── Lazy model loading ─────────────────────────────────────────────────────────
# Model is loaded on first call to _get_extractor() rather than at import time,
# so the module can be imported cheaply (e.g. in tests without a GPU/large model).

_EXTRACTOR = None


def _get_extractor() -> GLiNER2:
    """Return the GLiNER2 model, loading it on first use."""
    global _EXTRACTOR
    if _EXTRACTOR is None:
        log.info("Loading GLiNER2 model (fastino/gliner2-base-v1)...")
        _EXTRACTOR = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
        log.info("Model ready.")
    return _EXTRACTOR


#

def extract_info(item1_text: str) -> Optional[str]:
    """Run GLiNER2 on Item 1 text and return extracted description or None."""
    if not item1_text or not item1_text.strip():
        return None
    try:
        result = _get_extractor().extract_json(item1_text[:EXTRACTION_CHARS], SCHEMA)
        print(result)
        records = result.get("company", [])
        if not records:
            return None
        return records or None
    except Exception as e:
        log.debug(f"GLiNER2 error: {e}")
        return None



In [15]:
test_unit_1 = """ 
  Item 1 - Business General Information Lowe’s Companies, Inc. and subsidiaries (the Company or Lowe’s) is a Fortune 50 company and the world’s second largest home improvement retailer. As of January 29, 2010, we operated 1,710 stores, comprised of 1,694 stores across 50 U.S. states and 16 stores in Canada. Our 1,710 stores represent approximately 193 million square feet of retail selling space. Lowe’s was incorporated in North Carolina in 1952 and has been publicly held since 1961. The Company’s common stock is listed on the New York Stock Exchange - ticker symbol “LOW.” See Item 6, “Selected Financial Data,” of this Annual Report on Form 10-K, for historical revenues, profits and identifiable assets. For additional information about the Company’s performance and financial condition, see also Item 7 of this Annual Report on Form 10-K, “Management’s Discussion and Analysis of Financial Condition and Results of Operations.” Customers, Market and Competition Our Customers We serve homeowners, renters and Commercial Business Customers. Homeowners and renters primarily consist of do-it-yourself (DIY) customers and do-it-for-me (DIFM) customers who utilize our installed sales programs, as well as others buying for personal and family use. Commercial Business Customers include those who work in the construction, repair/remodel, commercial and residential property management, or business maintenance professions. Our Market Using the most recent comprehensive data available, which is from 2008, we estimate the size of the U.S. home improvement market to be approximately $630 billion annually, comprised of $492 billion of product demand and $138 billion of installed labor opportunity. Data from a variety of primary and secondary sources, including trade associations, government publications, industry participants and other sources was analyzed as the basis for our estimate. This data captures a wide range of categories relevant to our business, including major appliances and garden supplies. Based on the most recently available data, we believe that the size of the U.S. home improvement market decreased approximately 11% in 2009. There are many variables that impact consumer demand for the products and services we offer. Key indicators we monitor include employment, real disposable personal income, housing turnover, and home ownership levels. We also monitor demographic and societal trends that are indicators of home improvement industry growth. § Employment is an indicator of home improvement sales. The forecasted average unemployment rate of 10.0% for 2010 from the February 2010 Blue Chip Economic Indicators™ is higher than the 9.3% average seen in 2009 and suggests that Americans will continue to face challenging employment prospects this year. § Although real disposable personal income continues to grow, it is projected to grow at a slower pace for 2010 than the long-term average annual increase of 3.4%, during the period from 1960 to 2009. Real disposable personal income growth is forecasted to be 2.1% for calendar 2010, compared with 1.3% for calendar 2009, based on data from the February 2010 Blue Chip Economic Indicators™. § Housing turnover, which peaked in calendar year 2005, remains significantly below peak levels according to The National Association of Realtors®. However, recent data suggests that housing turnover in 2010 will increase over 2009. § According to the U.S. Census Bureau, while U.S. home ownership levels over the past year have continued their decline from 2007, they remain above their historical average. Home ownership provides an established customer base for home maintenance and repair projects. The vast majority of our customers are homeowners and most are not willing to let what is often their most valuable financial asset deteriorate. Currently, all of these indicators suggest continued weakness in consumer demand. In this challenging economic environment, we are balancing implementation of our long-term growth plans with our near-term focus on conserving capital and maintaining liquidity. Our Competition The home improvement retailing business includes many competitors. We compete with other chains of home improvement warehouse stores and lumberyards in most of our trade areas. We also compete with traditional hardware, plumbing, electrical and home supply retailers. In addition, we compete, with respect to some of our products, with general merchandise retailers, mail order firms, warehouse clubs, and online retailers. The principal competitive factors in our industry include location of stores, price and quality of merchandise, in-stock levels, merchandise assortment and presentation, and customer service. See further discussion of competition in

"""

test_result = extract_description(test_unit_1)

2026-03-18 11:18:26,421 INFO Loading GLiNER2 model (fastino/gliner2-base-v1)...
2026-03-18 11:18:26,665 INFO HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-18 11:18:26,703 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-18 11:18:26,845 INFO HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-18 11:18:26,888 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-18 11:18:27,004 INFO HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-18 11:18:27,045 INFO HTTP Request: H

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


2026-03-18 11:18:29,611 INFO HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-03-18 11:18:29,691 INFO Model ready.


{'company': [{'description': 'world’s second largest home improvement retailer', 'customers': 'homeowners', 'headquarters': 'North Carolina', 'founded': '1952', 'size': '1,710 stores', 'industry': 'home improvement'}]}


In [16]:
print(test_result)


world’s second largest home improvement retailer
